# Task 5: Mental Health Support Chatbot
## DevelopersHub Corporation — AI/ML Engineering Internship

**Name:** Muhammad Hassan Murad  
**DHC ID:** DHC-2360

---

### Objective
Build a chatbot that provides **supportive and empathetic responses** for stress, anxiety, and emotional wellness.

### Approach
This notebook uses **two complementary methods**:

| Method | Description |
|---|---|
| **Method A** | Fine-tuning `DistilGPT2` on the EmpatheticDialogues dataset (as specified in the task) |
| **Method B** | LLM prompt engineering with empathy-tuned system prompt (production-quality fallback) |

> **Why both?** Fine-tuning a small model like DistilGPT2 on empathetic dialogue teaches the full ML pipeline. The LLM method shows production-quality output. Both are shown for completeness.

---

## 1. Setup & Dependencies

In [ ]:
# Install all required libraries
!pip install transformers datasets torch accelerate anthropic --quiet
print("✅ Libraries installed.")

In [ ]:
import os
import torch
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Imports complete.")
print(f"   Device: {device} {'(GPU available 🚀)' if device == 'cuda' else '(CPU — training will be slower)'}")

---
## METHOD A: Fine-Tuning DistilGPT2 on EmpatheticDialogues
---

## 2. Load the EmpatheticDialogues Dataset

In [ ]:
from datasets import load_dataset

print("📡 Loading EmpatheticDialogues dataset from Hugging Face...")
dataset = load_dataset("empathetic_dialogues", trust_remote_code=True)

print(f"✅ Dataset loaded!")
print(f"   Train: {len(dataset['train'])} conversations")
print(f"   Validation: {len(dataset['validation'])} conversations")
print(f"   Test: {len(dataset['test'])} conversations")

# Preview a sample
sample = dataset['train'][0]
print(f"\nSample conversation:")
print(f"  Emotion  : {sample['context']}")
print(f"  Prompt   : {sample['prompt']}")
print(f"  Response : {sample['utterance']}")

## 3. Data Preprocessing — Prepare Training Format

We format each conversation as:
```
<emotion>: {emotion} | Person: {prompt} | Support: {response}
```
This teaches the model to associate emotions with supportive responses.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilgpt2"
MAX_LENGTH = 128
TRAIN_SAMPLES = 3000   # Use subset for reasonable training time on CPU

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # DistilGPT2 has no pad token


def format_sample(example):
    """
    Format each dialogue sample into a single training string.
    Template designed to teach empathetic, supportive responses.
    """
    text = (
        f"<emotion>: {example['context']} | "
        f"Person: {example['prompt'].strip()} | "
        f"Support: {example['utterance'].strip()}"
        f"{tokenizer.eos_token}"
    )
    return {"text": text}


def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


# Format and tokenize — use a subset for faster training
print(f"Formatting {TRAIN_SAMPLES} training samples...")
train_raw = dataset['train'].select(range(TRAIN_SAMPLES))
val_raw   = dataset['validation'].select(range(500))

train_formatted = train_raw.map(format_sample)
val_formatted   = val_raw.map(format_sample)

train_tokenized = train_formatted.map(tokenize, batched=True,
                                       remove_columns=train_formatted.column_names)
val_tokenized   = val_formatted.map(tokenize, batched=True,
                                     remove_columns=val_formatted.column_names)

train_tokenized.set_format(type='torch')
val_tokenized.set_format(type='torch')

print(f"✅ Tokenization complete.")
print(f"   Training samples : {len(train_tokenized)}")
print(f"   Validation samples: {len(val_tokenized)}")

# Show a formatted example
print(f"\nFormatted example:")
print(train_formatted[0]['text'][:200] + '...')

## 4. Load Pretrained Model & Configure Training

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded. Parameters: {total_params:,} ({total_params/1e6:.1f}M)")

# Data collator — handles dynamic padding during training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False  # mlm=False for causal LM
)

# Training configuration
training_args = TrainingArguments(
    output_dir             = './mental_health_model',
    overwrite_output_dir   = True,
    num_train_epochs       = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    warmup_steps           = 100,
    weight_decay           = 0.01,
    learning_rate          = 5e-5,
    eval_strategy          = 'epoch',
    save_strategy          = 'epoch',
    load_best_model_at_end = True,
    logging_steps          = 100,
    report_to              = 'none',   # Disable wandb
    fp16                   = (device == 'cuda'),  # Use mixed precision on GPU
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_tokenized,
    eval_dataset  = val_tokenized,
    data_collator = data_collator,
)

print("\n✅ Trainer configured. Ready to fine-tune.")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")

## 5. Fine-Tune the Model

> **Note:** Training on CPU takes ~30–60 minutes for 3000 samples × 3 epochs.  
> On GPU (Colab/Kaggle), it takes ~5–10 minutes.  
> **Tip:** Run this on Google Colab with a free T4 GPU for best results.

In [ ]:
print("🚀 Starting fine-tuning...")
print("   (This may take a while on CPU — consider using Google Colab with GPU)")
print("=" * 60)

train_result = trainer.train()

print("=" * 60)
print("✅ Fine-tuning complete!")
print(f"   Training loss : {train_result.training_loss:.4f}")
print(f"   Steps         : {train_result.global_step}")
print(f"   Time taken    : {train_result.metrics.get('train_runtime', 0):.1f}s")

# Save the fine-tuned model
trainer.save_model('./mental_health_model')
tokenizer.save_pretrained('./mental_health_model')
print("\n✅ Model saved to './mental_health_model'")

## 6. Generate Responses with Fine-Tuned Model

In [ ]:
from transformers import pipeline

# Load fine-tuned model for inference
generator = pipeline(
    'text-generation',
    model='./mental_health_model',
    tokenizer='./mental_health_model',
    device=0 if device == 'cuda' else -1
)


def get_empathetic_response(emotion: str, user_message: str) -> str:
    """
    Generate an empathetic response using the fine-tuned model.
    """
    prompt = f"<emotion>: {emotion} | Person: {user_message} | Support:"

    output = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = output[0]['generated_text']
    # Extract only the Support part
    if "Support:" in generated:
        response = generated.split("Support:")[-1].strip()
        response = response.split(tokenizer.eos_token)[0].strip()
    else:
        response = generated[len(prompt):].strip()

    return response


# Test the fine-tuned model
test_cases = [
    ("anxious",   "I have an important exam tomorrow and I can't stop worrying."),
    ("sad",       "I feel so alone. Nobody really understands what I'm going through."),
    ("stressed",  "Work is overwhelming me and I don't know how to cope."),
]

print("=" * 60)
print("Fine-Tuned Model Responses")
print("=" * 60)

for emotion, message in test_cases:
    response = get_empathetic_response(emotion, message)
    print(f"\n🔵 Emotion  : {emotion}")
    print(f"👤 Person   : {message}")
    print(f"🤖 Support  : {response}")
    print("-" * 60)

---
## METHOD B: LLM-Powered Empathetic Chatbot (Prompt Engineering)

A production-quality mental health support chatbot using an LLM with a carefully engineered empathy-focused system prompt.

---

## 7. Empathy System Prompt Design

In [ ]:
import anthropic

MENTAL_HEALTH_SYSTEM_PROMPT = """
You are MindfulBot, a warm and empathetic mental wellness support companion.

## Your Core Purpose
Provide emotional support, validation, and gentle coping strategies for people
experiencing stress, anxiety, sadness, loneliness, or emotional overwhelm.

## Communication Style — ALWAYS follow these
- Lead with empathy and validation before any advice
- Use warm, gentle, non-judgmental language
- Speak as a caring, patient friend — never clinical or cold
- Reflect back what you hear: "It sounds like you're feeling..."
- Keep responses concise (3–5 sentences) and focused
- Ask one gentle follow-up question to encourage sharing
- Normalise emotions: "What you're feeling makes complete sense"

## What You Offer
- Active listening and emotional validation
- Simple, practical coping strategies (breathing, grounding, journaling)
- Encouragement and gentle reframing
- Psychoeducation about stress and anxiety in simple terms

## STRICT Safety Rules
1. NEVER diagnose any mental health condition
2. NEVER replace professional therapy or psychiatry
3. If someone expresses thoughts of self-harm or suicide, IMMEDIATELY:
   - Acknowledge their pain with deep compassion
   - Provide crisis resources (e.g., crisis helpline)
   - Strongly encourage professional help
4. If someone is in immediate danger, direct them to emergency services
5. Always remind users you're a supportive tool, not a therapist

## Tone Examples
Good: "That sounds really exhausting. It makes complete sense that you feel overwhelmed."
Bad: "You should try to think more positively."

Good: "I hear you — carrying all of that alone takes so much energy."
Bad: "Have you tried just relaxing?"
""".strip()

print("✅ Empathy system prompt designed.")

## 8. Mental Health Chatbot Engine

In [ ]:
# Crisis detection keywords
CRISIS_KEYWORDS = [
    r"\bsuicid\b", r"\bkill myself\b", r"\bend my life\b",
    r"\bself.?harm\b", r"\bcut myself\b", r"\bdon't want to live\b",
    r"\bwant to die\b", r"\bno reason to live\b"
]

CRISIS_RESPONSE = (
    "💙 I hear you, and I'm so glad you're talking about this. "
    "What you're going through sounds incredibly painful, and you deserve real support right now.\n\n"
    "Please reach out to a crisis line immediately — they're available 24/7 and free:\n"
    "  • Pakistan: Umang helpline 0317-4288665\n"
    "  • International: iCall +91-9152987821\n"
    "  • Crisis Text Line: Text HOME to 741741\n\n"
    "You are not alone in this. 💙"
)

import re

def check_crisis(text: str) -> bool:
    lowered = text.lower()
    return any(re.search(p, lowered) for p in CRISIS_KEYWORDS)


class MentalHealthChatbot:
    """
    Empathetic mental health support chatbot.
    Features:
    - Emotion-aware system prompt (Method B)
    - Crisis detection with immediate resources
    - Multi-turn conversation memory
    - Gentle, validating tone
    """

    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.history = []
        self.model = "claude-sonnet-4-20250514"

    def chat(self, user_message: str) -> str:
        # Crisis safety check first
        if check_crisis(user_message):
            return CRISIS_RESPONSE

        self.history.append({"role": "user", "content": user_message})

        try:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=300,
                system=MENTAL_HEALTH_SYSTEM_PROMPT,
                messages=self.history,
            )
            reply = response.content[0].text
            self.history.append({"role": "assistant", "content": reply})
            return reply
        except Exception as e:
            self.history.pop()
            return f"⚠️ Error: {str(e)}"

    def reset(self):
        self.history = []
        print("🔄 Session reset.")


print("✅ MentalHealthChatbot class defined.")

## 9. Test Runs — Method B

In [ ]:
# Initialize bot (requires ANTHROPIC_API_KEY environment variable)
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not API_KEY:
    print("⚠️ Set ANTHROPIC_API_KEY to run Method B.")
    print("   In terminal: export ANTHROPIC_API_KEY='your-key'")
    print("   In Jupyter:  %env ANTHROPIC_API_KEY=your-key")
else:
    bot = MentalHealthChatbot(api_key=API_KEY)

    test_messages = [
        "I've been feeling really anxious lately and I don't know why.",
        "Work has been overwhelming me. I feel like I'm drowning.",
        "I feel so alone. Like nobody really gets me.",
    ]

    print("=" * 60)
    print("MindfulBot — LLM-Powered Empathetic Responses")
    print("=" * 60)

    for msg in test_messages:
        bot.reset()
        print(f"\n👤 Person  : {msg}")
        print(f"\n💙 MindfulBot:\n{bot.chat(msg)}")
        print("-" * 60)

    # Test crisis detection
    print("\n🧪 Crisis Detection Test:")
    print(f"👤 Person  : I don't want to live anymore")
    print(f"\n💙 MindfulBot:\n{bot.chat('I dont want to live anymore')}")

## 10. Interactive CLI Mode

Uncomment the code below and run to have a live conversation with MindfulBot.

In [ ]:
# =============================================================================
# INTERACTIVE SESSION — uncomment and run
# =============================================================================

# if API_KEY:
#     bot.reset()
#     print("=" * 60)
#     print("💙 MindfulBot — Mental Wellness Support")
#     print("=" * 60)
#     print("I'm here to listen. Share what's on your mind.")
#     print("Type 'quit' to end the session.\n")
#
#     while True:
#         user_input = input("👤 You: ").strip()
#         if not user_input:
#             continue
#         if user_input.lower() in ['quit', 'exit', 'bye']:
#             print("\n💙 MindfulBot: Take care of yourself. Remember, it's okay to ask for help. 💙")
#             break
#         print(f"\n💙 MindfulBot: {bot.chat(user_input)}\n")

## 11. Results & Key Insights

In [ ]:
print("""
====================================================
RESULTS SUMMARY — Mental Health Support Chatbot
====================================================

METHOD A — Fine-Tuned DistilGPT2:
  • Model: DistilGPT2 (82M parameters)
  • Dataset: EmpatheticDialogues (Facebook AI)
    → 25,000 conversations across 32 emotions
  • Training: 3 epochs on 3,000 samples
  • Format: emotion + prompt → empathetic response
  • Limitation: Small model; responses can be repetitive.
    Better results with GPT-Neo or Mistral-7B.

METHOD B — LLM Prompt Engineering:
  • Production-quality empathetic responses
  • Carefully engineered system prompt with:
    → Role definition (MindfulBot)
    → Communication style rules
    → Crisis safety handling
    → Tone examples (good vs bad)
  • Crisis detection pre-filter with helpline resources
  • Multi-turn conversation memory

KEY DESIGN DECISIONS:
  1. Validate before advising — empathy first, solutions second
  2. Crisis safety is non-negotiable — always routed to resources
  3. Short responses (3–5 sentences) — less overwhelming
  4. One follow-up question — encourages continued sharing

NEXT STEPS:
  • Use GPT-Neo-1.3B or Mistral-7B for better fine-tuning results
  • Add emotion classification to detect user mood automatically
  • Build a Streamlit UI for friendlier interaction
  • Add safety logging for monitoring concerning conversations
====================================================
""")